In [ ]:
from IPython.display import clear_output

var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
    "$var"/keras_nightly-3.12.0.dev2025100703-py3-none-any.whl \
      "$var"/tifffile-*.whl \
      "$var"/imagecodecs-*.whl \
    --no-index \
    --find-links "$var"

# Replace 'your-dataset-name' with the actual name of your uploaded Kaggle dataset
!pip install --no-index --find-links=/kaggle/input/datasets/sadek01/medicai-protobuf/wheels medicai protobuf
clear_output()

In [ ]:
import os, warnings

os.environ["KERAS_BACKEND"] = "torch"
os.environ["XLA_FLAGS"] = "--xla_gpu_force_compilation_parallelism=1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 
warnings.filterwarnings('ignore')

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ls checkpoints

In [ ]:
root_dir = "/kaggle/input/vesuvius-npy"
images_dir = f"{root_dir}/train_images"
labels_dir = f"{root_dir}/train_labels"
test_images_dir="/kaggle/input/vesuvius-challenge-surface-detection/test_images"
all_image_files = sorted(glob.glob(os.path.join(images_dir, "*.npy")))
all_label_files = sorted(glob.glob(os.path.join(labels_dir, "*.npy")))
test_image_files = sorted(glob.glob(os.path.join(test_images_dir, "*.tif")))

# number of validation samples
val_count = 6

# Split by slicing
train_imgs = all_image_files[:-val_count]
val_imgs   = all_image_files[-val_count:]

train_lbls = all_label_files[:-val_count]
val_lbls   = all_label_files[-val_count:]

print("Train images:", len(train_imgs), len(train_lbls))
print("Val images:  ", len(val_imgs), len(val_lbls))
print("Test_images: ", len(test_image_files) )

In [ ]:
import keras
from keras import ops

import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.callbacks import Callback

import medicai
from medicai.transforms import (
    Compose,
    NormalizeIntensity,
    ScaleIntensityRange,
    Resize,
    RandShiftIntensity,
    RandRotate90,
    RandRotate,
    RandFlip,
    RandCutOut,
    RandSpatialCrop
)
from medicai.layers import ResizingND
from medicai.models import (
    UNet, SegFormer, TransUNet, SwinUNETR, UPerNet, ConvNeXtV2Tiny, UNETRPlusPlus
)
from medicai.losses import (
    SparseDiceCELoss, SparseTverskyLoss, SparseCenterlineDiceLoss
)
from medicai.metrics import SparseDiceMetric
from medicai.callbacks import SlidingWindowInferenceCallback
from medicai.utils import SlidingWindowInference
from medicai.utils import soft_skeletonize

In [ ]:
keras.version(), keras.config.backend(), pl.__version__, medicai.version()

# Data Loader

In [ ]:
input_shape=(64, 128, 128)
batch_size=1
num_classes=3

# Total npy file - validation npy file
num_samples = 786 - val_count
epochs = 10

In [ ]:
# Assuming your dataset is named 'vesuvius-core-wheels'
!pip install --no-index --find-links=/kaggle/input/datasets/sadek01/wheelse/wheels batchgenerators monai

In [ ]:
def train_transformation(image, label):
    data = {"image": image, "label": label}
    pipeline = Compose([
        ## Geometric transformation
        RandFlip(keys=["image", "label"], spatial_axis=[0, 1, 2], prob=0.5),

        RandSpatialCrop(
            keys=["image", "label"],
            roi_size=input_shape,
            random_center=True,
            random_size=False,
            invalid_label=2,         
            min_valid_ratio=0.5,     
            max_attempts=8
        ),


        ## Z-score norm
        NormalizeIntensity(
            keys=["image"], 
            nonzero=True,
            channel_wise=False
        ),

        ## Intensiry transformation
        RandShiftIntensity(
            keys=["image"], offsets=0.10, prob=0.5
        ),
        

    ])
    result = pipeline(data)
    return (
        ops.convert_to_numpy(result["image"]), 
        ops.convert_to_numpy(result["label"])
    )


def val_transformation(image, label=None):
    if label is not None:
        data = {"image": image, "label": label}
    else:
        data={"image": image}
    pipeline = Compose([
        NormalizeIntensity(
            keys=["image"], 
            nonzero=True,
            channel_wise=False
        ),
    ])
    result = pipeline(data)
    if label is not None:
        return (
            ops.convert_to_numpy(result["image"]), 
            ops.convert_to_numpy(result["label"])
        )
    else :
                return (
            ops.convert_to_numpy(result["image"])
        )
        

In [ ]:
import tifffile
import glob
import os

class VesuviusDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, label_paths=None, mode='train'):
        # For test mode, image_paths should be a list of DIRECTORIES 
        # (each directory containing the .tif slices)
        self.images = image_paths
        self.labels = label_paths
        self.mode = mode

    def __len__(self):
        return len(self.images)

    def load_npy(self, path):
        array = np.load(path)   
        array = array[..., None]
        array = array.astype(np.float32)
        return array

    def load_volume_from_tif(self, file_path):
        vol = tifffile.imread(file_path)
        vol = vol.astype(np.float32)
        vol = vol[..., None]

        return vol

    def __getitem__(self, idx):
        # 1. Load Image
        if self.mode == 'test':
            # image_paths[idx] is now the path to your 1407735.tif file
            image = self.load_volume_from_tif(self.images[idx])
            
            # The rest of your logic remains the same
            
            image = val_transformation(image)
            image = torch.from_numpy(image).float().permute(3, 0, 1, 2)
            return image
            
        else:
            # Load actual .npy for train/val
            image = self.load_npy(self.images[idx])
            label = self.load_npy(self.labels[idx])
            
            if self.mode == 'train':
                image, label = train_transformation(image, label)
            else:
                image, label = val_transformation(image, label)

            image = torch.from_numpy(image).float().permute(3, 0, 1, 2)
            label = torch.from_numpy(label).float().permute(3, 0, 1, 2)
            
            return image, label

In [ ]:
from monai.data import Dataset as MonaiDataset   

In [ ]:
class VesuviusDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_imgs=None, 
        train_lbls=None, 
        val_imgs=None, 
        val_lbls=None,
        test_imgs=None, # Added for test
        train_batch_size=2,
        val_batch_size=1,
        test_batch_size=1, # Added for test
        num_workers=4,
    ):
        super().__init__()
        self.train_imgs = train_imgs
        self.train_lbls = train_lbls
        self.val_imgs = val_imgs
        self.val_lbls = val_lbls
        self.test_imgs = test_imgs
        
        self.train_batch_size = train_batch_size
        self.val_batch_size = val_batch_size
        self.test_batch_size = test_batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        # stage can be 'fit', 'validate', 'test', or 'predict'
        if stage == "fit" or stage is None:
            self.train_ds = VesuviusDataset(
                self.train_imgs, self.train_lbls, mode='train'
            )
            self.val_ds = VesuviusDataset(
                self.val_imgs, self.val_lbls, mode='val'
            )

        if stage == "predict" or stage is None:
            if self.test_imgs is not None:
                self.test_ds = VesuviusDataset(
                    self.test_imgs, label_paths=None, mode='test'
                )

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_ds, 
            batch_size=self.train_batch_size, 
            shuffle=True,
            num_workers=self.num_workers, 
            pin_memory=True
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_ds, 
            batch_size=self.val_batch_size, 
            shuffle=False,
            num_workers=self.num_workers, 
            pin_memory=True
        )

    def predict_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_ds, 
            batch_size=self.test_batch_size, 
            shuffle=False,
            num_workers=self.num_workers, 
            pin_memory=True
        )

In [ ]:
import os
# Clone the repository


# Add the repository to your Python path so we can import from it
import sys
sys.path.append('/kaggle/input/datasets/sadek01/my-working-dir-backup/unetr_plus_plus')

import torch
import torch.nn as nn

# Add all necessary paths for the official repo imports to work
paths = [
    '/kaggle/input/datasets/sadek01/my-working-dir-backup/unetr_plus_plus',
    '/kaggle/input/datasets/sadek01/my-working-dir-backup/unetr_plus_plus/unetr_pp/network_architecture/synapse'
]
for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)

# Import the specific class
from unetr_pp_synapse import UNETR_PP

# Initialize with the 14-class settings from the Synapse checkpoint
model = UNETR_PP(
    in_channels=1,
    out_channels=14, 
    img_size=(64, 128, 128),
    feature_size=16,
    num_heads=4,
    depths=[3, 3, 3, 3],
    dims=[32, 64, 128, 256],
    do_ds=True,
)


in_ch = model.out1.conv.in_channels

model.out1.conv = nn.Conv3d(in_ch, 3, kernel_size=1, stride=1, bias=False)

# Ensure the new head is trainable
for param in model.out1.parameters():
    param.requires_grad = True

model.do_ds = False


# 5. Verify the configuration
print("--- Unfreeze Status ---")
trainable_count = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_count += 1
        # Optional: print(f"Unfrozen: {name}") 

print(f"Total Trainable Parameter Tensors: {trainable_count}")
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")



# Lightning Module

In [ ]:
# loss function
dice_ce_loss_fn = SparseDiceCELoss(
    from_logits=True, 
    num_classes=num_classes,
    ignore_class_ids=2,
)
cldice_loss_fn = SparseCenterlineDiceLoss(
    from_logits=True, 
    num_classes=num_classes,
    target_class_ids=1,
    ignore_class_ids=2,
    iters=5, # ideal to set 20-50 - computationally expensive
)

def combined_loss(dice_ce_fn, cldice_fn):
    def loss_fn(y_true, y_pred):
        loss_dice_ce = dice_ce_fn(y_true, y_pred)
        loss_cldice = cldice_fn(y_true, y_pred)
        return loss_dice_ce + loss_cldice
    return loss_fn

In [ ]:
from monai.inferers import sliding_window_inference
import torch

In [ ]:
class VesuviusModel(pl.LightningModule):
    def __init__(
        self,
        model,
        learning_rate=1e-4,
        sw_batch_size=1,
        sw_overlap=0.5,
        num_classes=3,
        input_shape=(64, 128, 128)
    ):
        super().__init__()
        self.model = model
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.sw_batch_size = sw_batch_size
        self.sw_overlap = sw_overlap
        
        # Loss functions (using the global instances you defined)
        self.train_criterion = combined_loss(
            dice_ce_loss_fn, cldice_loss_fn
        )
        self.val_criterion = dice_ce_loss_fn

        # Metric functions
        self.train_dice_score = SparseDiceMetric(
            from_logits=True,
            num_classes=num_classes,
            ignore_class_ids=2,
            name='dice',
        )
        self.val_dice_score = SparseDiceMetric(
            from_logits=True,
            num_classes=num_classes,
            ignore_class_ids=2,
            name='val_dice',
        )
        
        # Sliding window inferer
        self.sliding_window_inferer = SlidingWindowInference(
            self.model,
            num_classes=self.num_classes,
            roi_size=input_shape,
            sw_batch_size=self.sw_batch_size,
            overlap=self.sw_overlap,
        )
        self.save_hyperparameters(ignore=['model'])

    def forward(self, x):
        # Model returns PyTorch format: [B, 3, 64, 128, 128]
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks = batch 
        
        # 1. Forward Pass
        logits = self.forward(images)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
            
        # 2. BRIDGE: Convert to MedicAI format (NDHWC + Softmax)
        # Permute: [B, C, D, H, W] -> [B, D, H, W, C]
        prob = logits.permute(0, 2, 3, 4, 1)
        
        
        # 3. BRIDGE: Convert Masks (B, 1, D, H, W) -> (B, D, H, W, 1)
        y_true = masks.permute(0, 2, 3, 4, 1).float()
        
        # 4. Calculate Loss & Metrics
        loss = self.train_criterion(y_true, prob)
        self.train_dice_score.update_state(y_true, prob.detach())
 
        # Logging
        current_dice_score = self.train_dice_score.result()
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_dice', current_dice_score, on_step=True, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        image, mask = batch # image: [B, 1, D, H, W]
        
        # 1. MONAI Sliding Window (Handles tiling on GPU)
        # We pass self.model directly; MONAI calls model(patch)
        output = sliding_window_inference(
            inputs=image, 
            roi_size=(64, 128, 128), # <-- Use your actual patch dimensions here
            sw_batch_size=self.sw_batch_size, 
            predictor=self.model,
            overlap=self.sw_overlap,
            mode="gaussian",
        )
        
        # 2. Convert to Probabilities and permute to NDHWC
        prob =output.permute(0, 2, 3, 4, 1)
        y_true = mask.permute(0, 2, 3, 4, 1).float()
        
        # 3. Update Metrics and Loss
        loss = self.val_criterion(y_true, prob)
        self.val_dice_score.update_state(y_true, prob.detach())
    
        # Logging
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_dice', self.val_dice_score.result(), on_epoch=True, prog_bar=True)
        
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.parameters()), 
            lr=self.learning_rate,
            weight_decay=1e-5
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2,
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_dice',
                'frequency': 1
            }
        }

    def on_train_epoch_end(self):
        self.train_dice_score.reset_state()
        import gc
        gc.collect()
        torch.cuda.empty_cache()
    def on_train_batch_end(self, outputs, batch, batch_idx):
        # Clear the loss reference from the outputs dict to ensure it's freed
        if "loss" in outputs:
            outputs["loss"] = outputs["loss"].detach()
        import gc
        gc.collect()
            
        
        torch.cuda.empty_cache()
                

    def on_validation_epoch_end(self):
        self.val_dice_score.reset_state()
        torch.cuda.empty_cache()
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        # 1. Get image
        image = batch[0] if isinstance(batch, (list, tuple)) else batch
        
        # 2. MONAI Sliding Window Inference
        output = sliding_window_inference(
            inputs=image, 
            roi_size=self.hparams.input_shape, 
            sw_batch_size=self.sw_batch_size, 
            predictor=self.model,
            overlap=self.sw_overlap,
            mode="gaussian",
        )
        
        # 3. POST-PROCESSING: Filter for classes 0 and 1
        # output is [B, C, D, H, W] -> Slicing C to keep only indices 0 and 1
        # This effectively ignores class 2 entirely
        output_filtered = output[:, :2, ...] 
        
        # 4. Get the best class among the two
        prob = torch.softmax(output_filtered, dim=1)
        mask = torch.argmax(prob, dim=1) # Results in [B, D, H, W] with values {0, 1}
        
        # 5. Return as NumPy
        return mask.squeeze().detach().cpu().numpy().astype(np.uint8)   

**Custom Print Callback**

In [ ]:
class MetricLoggerCallback(Callback):
    def __init__(self, print_every_n_steps=20):
        super().__init__()
        self.print_every_n_steps = print_every_n_steps

    def on_train_batch_end(
        self, trainer, pl_module, outputs, batch, batch_idx
    ):
        global_step = trainer.global_step
        if global_step > 0 and (global_step % self.print_every_n_steps == 0):
            log_metrics = trainer.callback_metrics 
            loss = log_metrics.get('train_loss_step')
            dice = log_metrics.get('train_dice_step')
            print(
                f"[step {global_step}] batch log | "
                f"loss: {loss:.4f}, dice: {dice:.4f}"
            )

    def on_train_epoch_end(self, trainer, pl_module):
        log_metrics = trainer.callback_metrics 
        loss = log_metrics.get('train_loss_epoch')
        dice = log_metrics.get('train_dice_epoch')
        print(
            f"train loss: {loss:.4f}, train dice: {dice:.4f}"
        )
        print("-" * 50)

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss_key = 'val_loss' 
        val_dice_key = 'val_dice'
        val_loss = (
            trainer.callback_metrics.get(val_loss_key) or 
            trainer.callback_metrics.get(f'{val_loss_key}_epoch')
        )
        val_dice = (
            trainer.callback_metrics.get(val_dice_key) or 
            trainer.callback_metrics.get(f'{val_dice_key}_epoch')
        )
        print(
            f"epoch {trainer.current_epoch + 1:02d} completed: "
            f"val loss: {val_loss:.4f}, val dice: {val_dice:.4f}"
        )

In [ ]:
checkpoint_dir = '/kaggle/working/checkpoints'

# Create the directory if it doesn't exist
os.makedirs(checkpoint_dir, exist_ok=True)
latest_checkpoint = ModelCheckpoint(
    dirpath='/kaggle/working/checkpoints',
    filename='latest-{epoch:02d}',
    save_top_k=1,
    monitor='epoch',  # Monitoring 'epoch' with mode='max' ensures it always updates
    mode='max',
    save_last=True     # This will now correctly maintain a 'last.ckpt' for every epoch
)
print(f"✅ Directory ready at: {checkpoint_dir}")
callbacks = [
    MetricLoggerCallback(),
    ModelCheckpoint(
        dirpath='/kaggle/working/checkpoints',  # <--- This locks the folder
        monitor='val_dice',
        mode='max',
        save_top_k=2,
        save_last=True,
        filename='best-{epoch:02d}-{val_dice:.4f}',
        auto_insert_metric_name=False # Makes filenames cleaner
    ),latest_checkpoint,
    LearningRateMonitor(logging_interval='epoch')
]

In [ ]:
# create data module
data_module = VesuviusDataModule(
    train_imgs=train_imgs,
    train_lbls=train_lbls,
    val_imgs=val_imgs,
    val_lbls=val_lbls,
    train_batch_size=4,
    val_batch_size=1,
    num_workers=4,
)

In [ ]:
# create the Lightning model
lightning_model = VesuviusModel(
    model=model,
    learning_rate=1e-4,
    sw_batch_size=1,
    sw_overlap=0.25
)

In [ ]:
trainer = pl.Trainer(
    max_epochs=500,
    precision="16-mixed",
    callbacks=callbacks,
    accelerator='auto',
    devices='auto',
    log_every_n_steps=20,
    accumulate_grad_batches=1,
    check_val_every_n_epoch=1,
    enable_progress_bar=False,
    num_sanity_val_steps=0
)

In [ ]:
import torch

checkpoint = torch.load("/kaggle/working/checkpoints/last.ckpt", map_location="cpu")
print(f"Epoch: {checkpoint['epoch']}")
print(f"Global Step: {checkpoint['global_step']}")

In [ ]:
checkpoint_path = "/kaggle/input/datasets/sadek01/model-checkpoint/best-125-0.7599.ckpt"

# 1. Initialize fresh instance
lightning_model = VesuviusModel(
    model=model,
    learning_rate=1e-4,
    sw_batch_size=1,
    sw_overlap=0.25
)


trainer.fit(lightning_model, datamodule=data_module,ckpt_path=checkpoint_path)


In [ ]:
trainer.fit(lightning_model, datamodule=data_module)